# Dataset generation and validation splits

Builds every dataset the GRU experiments will use, **once**, so that all models share the same folds.

| variant | what it is |
|---|---|
| `raw_{uk,ru,all}` | original sentences, with `fold` (0..K-1) and a `holdout` split |
| `long_{uk,ru,all}` | sentences glued into Telegram-style posts whose lengths follow the **test** length distribution |
| `val_labeling_sample` | the 25 in-domain rows from the competition (tiny, but closest to the test) |

**Leakage rules**
- ru: the group is `doc_id` (sentences of one document never cross folds).
- uk: the group is the hash of the text (exact duplicates stay together). The official `is_valid == 1` rows are the holdout.
- Dev rows whose text also appears in the holdout are dropped.
- `long_*` posts are built **inside** one fold or holdout, and inherit its label. A post never mixes folds.
- `fold` is a deterministic hash of the group, so re-running gives identical splits.

In [1]:
import ast, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------- CONFIG -------------------------------
IN_DIR       = Path("Data")            # where the competition files live
UK_CSV       = IN_DIR / "uk_geo_dataset.csv"
RU_CSV       = IN_DIR / "ru_geo_dataset.csv"
TEST_CSV     = IN_DIR / "test.csv"           # only used for its text-length distribution
LABELING_CSV = IN_DIR / "labeling_sample.csv"
OUT_DIR      = Path("data/splits")

SEED            = 42
N_FOLDS         = 5
UK_NROWS        = None     # None = all ~1M rows (set e.g. 100_000 for a quick dry run)
RU_KEEP_FRAC    = 0.25     # fraction of ru DOCUMENTS to keep (8M rows is too many for one GPU)
RU_HOLDOUT_FRAC = 0.01     # fraction of kept ru documents reserved as holdout (uk uses is_valid)

# --- "long" (post-like) dataset ---
MAX_CHARS       = 2500     # hard cap for a glued post
EMOJI_PREFIX_P  = 0.15     # chance a post starts with an emoji marker (Telegram style)
FOOTER_P        = 0.0      # chance of a channel footer line; 0 = off (see markdown below)

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Loading

In [2]:
HASH = lambda n: f"{SEED + n:016d}"          # pandas wants a 16-char hash key
KEY_FOLD, KEY_KEEP, KEY_HOLD = HASH(0), HASH(1), HASH(2)

def hkey(values, key):
    """Stable 64-bit hash of strings (same input -> same output, any machine)."""
    return pd.util.hash_pandas_object(pd.Series(values, dtype=object), index=False, hash_key=key).to_numpy()

def frac_mask(values, key, frac):
    """Deterministic 'random' subset: True for ~frac of distinct values."""
    return (hkey(values, key) % 10_000) < int(frac * 10_000)

def parse_markers(s, n):
    """'[(82, 88), ...]' -> [[82, 88], ...], clipped to the text length n."""
    if s == "[]":
        return []
    return [[a, min(b, n)] for a, b in ast.literal_eval(s) if 0 <= a < min(b, n)]

def load_lang(path, lang, keep_frac=1.0, nrows=None, chunksize=250_000):
    parts = []
    for ch in pd.read_csv(path, chunksize=chunksize, nrows=nrows):
        if lang == "ru":
            ch["group"] = "ru:" + ch["doc_id"].astype(str)
            ch = ch[frac_mask(ch["group"], KEY_KEEP, keep_frac)] if keep_frac < 1 else ch
        else:
            ch["group"] = np.char.add("uk:", hkey(ch["text"].astype(str), KEY_FOLD).astype(str))
            ch["doc_id"], ch["sent_id"] = -1, -1
        for c in ("loc_markers", "org_markers", "per_markers"):   # parse only the rows we keep
            ch[c] = [parse_markers(s, len(t)) for s, t in zip(ch[c], ch["text"].astype(str))]
        ch["text"] = ch["text"].astype(str)
        if lang == "uk":
            ch["sent_id"] = ch.index
        ch["lang"] = lang
        parts.append(ch)
    df = pd.concat(parts, ignore_index=True)
    return df

uk = load_lang(UK_CSV, "uk", nrows=UK_NROWS)
print("uk", len(uk), "rows;", uk["is_valid"].sum(), "official valid rows")
ru = load_lang(RU_CSV, "ru", keep_frac=RU_KEEP_FRAC)
print("ru", len(ru), "rows;", ru["doc_id"].nunique(), "documents kept")

uk 1010000 rows; 10000 official valid rows
ru 1999684 rows; 184223 documents kept


## 2. Folds and holdout

`fold = hash(group) % K`, and `-1` means holdout. Duplicates of holdout texts are removed from the dev pool.

In [3]:
def assign_split(df, lang):
    if lang == "uk":
        hold = df["is_valid"].to_numpy() == 1
    else:
        hold = frac_mask(df["group"], KEY_HOLD, RU_HOLDOUT_FRAC)
    fold = (hkey(df["group"], KEY_FOLD) % N_FOLDS).astype(int)
    fold[hold] = -1
    df = df.assign(fold=fold, split=np.where(hold, "holdout", "dev"))
    # exact-text leakage: a dev sentence that also exists in the holdout
    hold_texts = set(df.loc[hold, "text"])
    leak = (~hold) & df["text"].isin(hold_texts).to_numpy()
    print(f"{lang}: dropping {leak.sum()} dev rows whose text is also in holdout")
    return df[~leak].reset_index(drop=True)

COLS = ["text", "loc_markers", "org_markers", "per_markers",
        "lang", "group", "doc_id", "sent_id", "split", "fold"]

raw = {"uk": assign_split(uk, "uk")[COLS], "ru": assign_split(ru, "ru")[COLS]}
del uk, ru

for k, d in raw.items():
    d["locations"] = [[t[a:b] for a, b in m] for t, m in zip(d["text"], d["loc_markers"])]
    d["n_loc"] = d["loc_markers"].map(len)

uk: dropping 52 dev rows whose text is also in holdout
ru: dropping 10596 dev rows whose text is also in holdout


## 3. Test-length distribution

The long dataset draws its target post length from the real test texts, so the model trains on the length it will be scored on.

In [4]:
if TEST_CSV.exists():
    TEST_LENS = pd.read_csv(TEST_CSV)["text"].astype(str).str.len().clip(lower=20, upper=MAX_CHARS).tolist()
else:
    print("test.csv not found -> lognormal fallback (median 264 chars)")
    TEST_LENS = np.clip(np.random.default_rng(SEED).lognormal(np.log(264), 0.9, 3000), 20, MAX_CHARS).astype(int).tolist()
print(len(TEST_LENS), "target lengths; quantiles:", np.percentile(TEST_LENS, [10, 25, 50, 75, 90, 99]).round())

477 target lengths; quantiles: [  50.  128.  264.  517.  850. 1969.]


## 4. Building the "long" posts

- **ru**: sentences of one document are glued in their real order (`sent_id`), so it is genuine context.
- **uk**: no document structure exists, so random sentences from the **same fold** are glued.
- Separators are `" "`, `"\n"` or `"\n\n"`. Entity offsets are shifted by the position of each sentence.
- Optional decoration: an emoji prefix. A channel footer exists too, but `FOOTER_P=0` by default. The 25 labeled rows show footers such as "Київ — пряма трансляція" are *not* labeled as locations, but that is only an observation from 25 rows. Turn it on only if the validation backs it up.

In [5]:
EMOJI = ["❗️", "🔥", "📍", "🚨", "‼️", "⚡️", "🔴", "🇺🇦", "🚧", "💥"]
FOOTERS = ["Київ — пряма трансляція 👈\nНадіслати новину", "Підписатися    |    Запропонувати новину"]
SEPS, SEP_W = [" ", "\n", "\n\n"], [0.5, 0.25, 0.25]

def segments(df, lang, rnd):
    """Index arrays; each segment is glued only with itself."""
    if lang == "ru":                                   # one segment per document, in sentence order
        order = df.sort_values(["doc_id", "sent_id"], kind="stable")
        return list(order.groupby("doc_id", sort=False).indices.values()), order.index.to_numpy()
    segs = []
    for f in sorted(df["fold"].unique()):               # uk: one shuffled segment per fold / holdout
        idx = df.index[df["fold"] == f].to_numpy().copy()
        rnd.shuffle(idx)
        segs.append(idx)
    return segs, None

def build_long(df, lang, seed):
    rnd = random.Random(seed)
    df = df.reset_index(drop=True)
    T, L, O, P = (df[c].tolist() for c in ("text", "loc_markers", "org_markers", "per_markers"))
    LOCS, SPLIT, FOLD, GRP, DOC = (df[c].tolist() for c in ("locations", "split", "fold", "group", "doc_id"))

    segs, order = segments(df, lang, rnd)
    if lang == "ru":                                    # groupby.indices gives positions inside `order`
        segs = [order[s] for s in segs]

    out = []
    for seg in segs:
        i, n = 0, len(seg)
        while i < n:
            target = rnd.choice(TEST_LENS)
            text, loc, org, per, src_loc = "", [], [], [], []
            if rnd.random() < EMOJI_PREFIX_P:
                text = rnd.choice(EMOJI) + " "
            first = seg[i]
            while i < n:
                j = seg[i]
                sep = "" if (text == "" or text.endswith(" ")) else rnd.choices(SEPS, SEP_W)[0]
                if text and len(text) + len(sep) + len(T[j]) > MAX_CHARS:
                    break
                pos = len(text) + len(sep)
                text = text + sep + T[j]
                loc += [[a + pos, b + pos] for a, b in L[j]]
                org += [[a + pos, b + pos] for a, b in O[j]]
                per += [[a + pos, b + pos] for a, b in P[j]]
                src_loc += LOCS[j]
                i += 1
                if len(text) >= target:
                    break
            if FOOTER_P and rnd.random() < FOOTER_P:
                text += "\n\n" + rnd.choice(FOOTERS)
            out.append((text, loc, org, per, lang, GRP[first], DOC[first], SPLIT[first], FOLD[first], src_loc))
    res = pd.DataFrame(out, columns=["text", "loc_markers", "org_markers", "per_markers", "lang",
                                     "group", "doc_id", "split", "fold", "_src_loc"])
    res["locations"] = [[t[a:b] for a, b in m] for t, m in zip(res["text"], res["loc_markers"])]
    res["n_loc"] = res["loc_markers"].map(len)
    return res

long_ = {}
for lang in ("uk", "ru"):
    long_[lang] = build_long(raw[lang], lang, SEED + (1 if lang == "uk" else 2))
    d = long_[lang]
    assert (d["locations"] == d["_src_loc"]).all(), "entity offsets were shifted incorrectly"
    d.drop(columns="_src_loc", inplace=True)
    print(lang, len(raw[lang]), "sentences ->", len(d), "posts; median chars", int(d["text"].str.len().median()))

uk 1009948 sentences -> 229735 posts; median chars 327
ru 1989088 sentences -> 656460 posts; median chars 284


## 5. Combined variants and saving

In [6]:
raw["all"]   = pd.concat([raw["uk"], raw["ru"]], ignore_index=True)
long_["all"] = pd.concat([long_["uk"], long_["ru"]], ignore_index=True)

def save(df, name):
    df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)

for k in ("uk", "ru", "all"):
    save(raw[k],   f"raw_{k}")
    save(long_[k], f"long_{k}")

# in-domain validation set (only 25 rows, but it comes from the test distribution)
lab = pd.read_csv(LABELING_CSV)
lab["text"] = lab["text"].astype(str)
lab["loc_markers"] = [parse_markers(s, len(t)) for s, t in zip(lab["loc_markers"], lab["text"])]
lab["locations"] = [[t[a:b] for a, b in m] for t, m in zip(lab["text"], lab["loc_markers"])]
lab.to_parquet(OUT_DIR / "val_labeling_sample.parquet", index=False)
print(sorted(p.name for p in OUT_DIR.glob("*.parquet")))

['long_all.parquet', 'long_ru.parquet', 'long_uk.parquet', 'raw_all.parquet', 'raw_ru.parquet', 'raw_uk.parquet', 'val_labeling_sample.parquet']


## 6. Sanity checks

In [7]:
# 6.1 no group is in two folds, and holdout groups never appear in dev
for k, d in raw.items():
    assert d.groupby("group")["fold"].nunique().max() == 1, f"group leaks across folds in raw_{k}"
    hold_groups = set(d.loc[d["split"] == "holdout", "group"])
    assert not (set(d.loc[d["split"] == "dev", "group"]) & hold_groups), f"holdout groups in dev ({k})"
print("OK: groups never cross folds / holdout")

# 6.2 the same sentence is never in train-fold and val-fold of a long post dataset
for k, d in long_.items():
    assert (d["fold"] >= 0).eq(d["split"] == "dev").all()
print("OK: long posts inherit split/fold consistently")

# 6.3 summary table
rows = []
for name, dsets in (("raw", raw), ("long", long_)):
    for k, d in dsets.items():
        for f, g in d.groupby("fold"):
            rows.append(dict(variant=f"{name}_{k}", fold=f, rows=len(g),
                             with_loc=round((g["n_loc"] > 0).mean(), 3),
                             loc_per_row=round(g["n_loc"].mean(), 3),
                             median_chars=int(g["text"].str.len().median())))
summary = pd.DataFrame(rows)
summary.pivot(index="fold", columns="variant", values="rows")

OK: groups never cross folds / holdout
OK: long posts inherit split/fold consistently


variant,long_all,long_ru,long_uk,raw_all,raw_ru,raw_uk
fold,,,,,,
-1,8785,6486,2299,29802,19802,10000
0,175604,130003,45601,594274,393915,200359
1,175510,130342,45168,593613,394268,199345
2,175400,129995,45405,593364,393787,199577
3,175320,129656,45664,594098,394056,200042
4,175576,129978,45598,593885,393260,200625


In [8]:
summary.pivot(index="fold", columns="variant", values="with_loc")   # rate of rows with >=1 location; should be flat across folds

variant,long_all,long_ru,long_uk,raw_all,raw_ru,raw_uk
fold,,,,,,
-1,0.571,0.574,0.560,0.287,0.315,0.230
0,0.571,0.567,0.583,0.283,0.310,0.231
1,0.569,0.565,0.580,0.283,0.310,0.231
2,0.569,0.564,0.581,0.282,0.308,0.232
3,0.566,0.561,0.578,0.281,0.306,0.231
4,0.564,0.560,0.577,0.282,0.308,0.230


In [9]:
# 6.4 does the long dataset really look like the test in length?
q = [10, 25, 50, 75, 90, 99]
cmp = pd.DataFrame({"test": np.percentile(TEST_LENS, q)}, index=[f"p{x}" for x in q])
for k, d in long_.items():
    cmp[f"long_{k}"] = np.percentile(d["text"].str.len(), q)
for k, d in raw.items():
    cmp[f"raw_{k}"] = np.percentile(d["text"].str.len(), q)
cmp.round(0).astype(int)

,test,long_uk,long_ru,long_all,raw_uk,raw_ru,raw_all
p10,50,112,111,111,51,54,53
p25,128,188,171,175,66,77,73
p50,264,327,284,294,91,111,104
p75,517,586,484,506,128,151,144
p90,850,922,737,778,170,195,188
p99,1969,2082,1315,1577,238,300,281


In [10]:
# 6.5 offsets: do spans of the labeling sample and of a long post cut out the entity text?
print(lab[["locations"]].head(5).to_string())
d = long_["all"]
ex = d[d["n_loc"] >= 3].sample(1, random_state=0).iloc[0]
print(repr(ex["text"][:400]))
print("locations:", ex["locations"])

                                                                                            locations
0                                                                                                  []
1                                                                                  [Проспект Бандери]
2  [Київщині, Чорнобиль, Київ, Тетерів, Фастів, Біла Церква, Миронівка, Бориспіль, Яготин, Баришівка]
3                                                                                 [проспекті Івасюка]
4                                                                                      [Києва, Хелма]
'Как пишет Джорджевич в своем письме, в 2001 году он покинул страну, поскольку новое правительство хотело его арестовать для дачи показаний против Милошевича в военном трибунале ООН.\nДжорджевич заявляет, что не имел полномочий во время войны в Косово в 1999 году и отрицет наличие какой-либо договоренности с трибуналом ООН. "Все сообщения о своей договоренности с трибуналом в Гааге и н

## Using the files

```python
tr = pd.read_parquet("data/splits/long_all.parquet")
fold = 0
train, valid = tr[tr.fold != fold][tr.split == "dev"], tr[tr.fold == fold]
holdout      = tr[tr.split == "holdout"]            # only for the final check
```
Report CV on the folds, the holdout, and `val_labeling_sample` for each experiment.